<a href="https://colab.research.google.com/github/angielira/ALAL-Bootcamp-Build-your-Foundation/blob/main/Spam_NLP_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# The Machine That Doesn't Know What "LOVE" Means



Raw tweets → clean them → turn them into numbers → train a model → grade the model → use it on new text.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
# %matplotlib inline

In [ ]:
# df = pd.read_csv("/content/drive/MyDrive/Datasets/SMSSpamCollection.txt", sep ='\t', names=['label', 'message'])
url = "https://raw.githubusercontent.com/fenago/datasets/refs/heads/main/TheSocialDilemma.csv"
df = pd.read_csv(url, encoding='ISO-8859-1')  # or try encoding='latin1'
df.head()

,user_name,user_location,user_description,user_created,user_followers,user_friends,user_favourites,user_verified,date,text,hashtags,source,is_retweet,Sentiment
0,Mari Smith,"San Diego, California",Premier Facebook Marketing Expert | Social Med...,2007-09-11 22:22:51,579942,288625,11610,False,2020-09-16 20:55:33,@musicmadmarc @SocialDilemma_ @netflix @Facebo...,NaN,Twitter Web App,False,Neutral
1,Mari Smith,"San Diego, California",Premier Facebook Marketing Expert | Social Med...,2007-09-11 22:22:51,579942,288625,11610,False,2020-09-16 20:53:17,@musicmadmarc @SocialDilemma_ @netflix @Facebo...,NaN,Twitter Web App,False,Neutral
2,Varun Tyagi,"Goa, India",Indian | Tech Solution Artist & Hospitality Ex...,2009-09-06 10:36:01,257,204,475,False,2020-09-16 20:51:57,Go watch âThe Social Dilemmaâ on Netflix!\...,NaN,Twitter for iPhone,False,Positive
3,Casey Conway,"Sydney, New South Wales",Head of Diversity & Inclusion @RugbyAU | It's ...,2012-12-28 21:45:06,11782,1033,12219,True,2020-09-16 20:51:46,I watched #TheSocialDilemma last night. Iâm ...,['TheSocialDilemma'],Twitter for iPhone,False,Negative
4,Charlotte Paul,Darlington,Instagram Charlottejyates,2012-05-28 20:43:08,278,387,5850,False,2020-09-16 20:51:11,The problem of me being on my phone most the t...,['TheSocialDilemma'],Twitter for iPhone,False,Positive


In [ ]:
len(df)

20068

In [ ]:
df.isnull().sum()

,0
user_name,1
user_location,4208
user_description,1383
user_created,0
user_followers,0
user_friends,0
user_favourites,0
user_verified,0
date,0
text,0


In [ ]:
# Label
df['Sentiment'].unique()

array(['Neutral', 'Positive', 'Negative'], dtype=object)

In [ ]:
# How many tweets of each kind?
df['Sentiment'].value_counts()

,count
Sentiment,
Positive,9513
Neutral,6982
Negative,3573


In [ ]:
import pandas as pd

# Assuming df is your original dataframe
# First, separate the classes
positive_df = df[df['Sentiment'] == 'Positive']
negative_df = df[df['Sentiment'] == 'Negative']

# Downsample the majority class
positive_downsampled = positive_df.sample(n=len(negative_df), random_state=42)

# Combine the minority class with the downsampled majority class
balanced_df = pd.concat([positive_downsampled, negative_df])

# Shuffle the dataframe
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Validate the counts
print(balanced_df['Sentiment'].value_counts())

Sentiment
Negative    3573
Positive    3573
Name: count, dtype: int64


In [ ]:
df = balanced_df
df['Sentiment'].value_counts()

,count
Sentiment,
Negative,3573
Positive,3573


In [ ]:
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()
corpus = []

# Updated loop using iterrows()
for index, row in df.iterrows():
    review = re.sub('[^a-zA-Z]', ' ', row['text'])
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
corpus[:5]

['thesocialdilemma liter put spell us inform terrifi time must watch http co sifbmfc',
 'would recommend everi singl individu go digit wellb screen time page phone http co jbdbizsf g',
 'get social media thesocialdilemma',
 'finish watch social dilemma netflix wow thought provok documentari danger http co puanqk qke',
 'serious contempl delet social media watch thesocialdilemma']

In [ ]:
X = df['text']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

X_train.shape

(5002,)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
count_vect=CountVectorizer()
X_train_counts =count_vect.fit_transform(X_train)

print("Shape of count vectorizer", X_train_counts.shape)

from sklearn.feature_extraction.text import TfidfTransformer
tfidf_transformer = TfidfTransformer()
X_train_tfidf =tfidf_transformer.fit_transform(X_train_counts)

print("Shape of tfidf feature extraction",X_train_tfidf.shape)

Shape of count vectorizer (5002, 11200)
Shape of tfidf feature extraction (5002, 11200)


In [ ]:
type(X_train_tfidf)

scipy.sparse._csr.csr_matrix

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(solver='lbfgs')
clf.fit(X_train_tfidf, y_train)

LogisticRegression()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

text_clf = Pipeline([('tfidf', TfidfVectorizer()),
                     ('clf', LogisticRegression()),])
text_clf.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('clf', LogisticRegression())])

In [ ]:
predictions = text_clf.predict(X_test)

In [ ]:
from sklearn import metrics
print("Confusion Metrics\n",metrics.confusion_matrix(y_test,predictions), end="\n\n\n")

print("Classification Report\n",metrics.classification_report(y_test,predictions), end="\n\n\n")

print("Accuracy Score:", metrics.accuracy_score(y_test,predictions))

Confusion Metrics
 [[849 207]
 [143 945]]


Classification Report
               precision    recall  f1-score   support

    Negative       0.86      0.80      0.83      1056
    Positive       0.82      0.87      0.84      1088

    accuracy                           0.84      2144
   macro avg       0.84      0.84      0.84      2144
weighted avg       0.84      0.84      0.84      2144



Accuracy Score: 0.8367537313432836


In [ ]:
def predict_sentiment(text):
    # Preprocess the text: remove non-letters, convert to lowercase, split into words
    review = re.sub('[^a-zA-Z]', ' ', text)
    review = review.lower()
    review = review.split()

    # Remove stopwords and apply stemming
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)

    # Predict using the trained model
    prediction = text_clf.predict([review])
    return prediction

# Example usage
input_text = "I love New York"  # Replace with your own text
predicted_sentiment = predict_sentiment(input_text)
print("Predicted Sentiment:", predicted_sentiment)

Predicted Sentiment: ['Positive']


In [ ]:
def predict_sentiment(text):
    # Preprocess the text: remove non-letters, convert to lowercase, split into words
    review = re.sub('[^a-zA-Z]', ' ', text)
    review = review.lower()
    review = review.split()

    # Remove stopwords and apply stemming
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)

    # Predict using the trained model
    prediction = text_clf.predict([review])[0]                 # the label, e.g. 'Positive'
    probabilities = text_clf.predict_proba([review])[0]        # one probability per class, in text_clf.classes_ order
    confidence = probabilities.max()                           # the probability of the winning class
    return prediction, confidence


In [ ]:

# Example usage
input_text = "I hate New York"  # Replace with your own text
predicted_sentiment, confidence = predict_sentiment(input_text)
print(f"Predicted Sentiment: {predicted_sentiment} ({confidence:.0%} confident)")

Predicted Sentiment: Negative (61% confident)


In [ ]:
def predict_sentiment_probability(text):
    # Preprocess the text: remove non-letters, convert to lowercase, split into words
    review = re.sub('[^a-zA-Z]', ' ', text)
    review = review.lower()
    review = review.split()

    # Remove stopwords and apply stemming
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)

    # Predict using the trained model
    probabilities = text_clf.predict_proba([review])
    return probabilities

# Example usage
input_text = "I am neutral about New York"  # Replace with your own text
predicted_probabilities = predict_sentiment_probability(input_text)
print("Predicted Probabilities:", predicted_probabilities)

Predicted Probabilities: [[0.30268822 0.69731178]]
